# Employee Weekly Timesheet Report

This notebook contains the **complete working code** using sample `employee_df` and `timesheet_df`.

## Business rules

For every employee and every week:

- **Submitted** → submitted days `> 4`
- **Filled but not submitted** → records exist, but submitted days `<= 4`
- **Blank** → no timesheet record exists for that week

### Daily rule

An employee may have multiple timesheet rows on the same date.

For one employee/date:

- If **any** row has `Status = Submitted` → that day is `Submitted`
- Otherwise, if a record exists → `Filled but not submitted`
- If no record exists → `Blank`

Excel serial dates such as `46267` are converted using the Excel 1900 date system.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)


## 1. Sample Employee DataFrame

In [ ]:
employee_df = pd.DataFrame([
    {"EmpID": "E001", "Name": "Rahul", "Team": "Technical"},
    {"EmpID": "E002", "Name": "Amit",  "Team": "Technical"},
    {"EmpID": "E003", "Name": "Priya", "Team": "Technical"},
    {"EmpID": "E004", "Name": "Sneha", "Team": "Actuarial"},
    {"EmpID": "E005", "Name": "Vikas", "Team": "Actuarial"},
    {"EmpID": "E006", "Name": "Neha",  "Team": "HR"},
])

employee_df


## 2. Excel Serial Date Helper

In [ ]:
def excel_serial(date):
    """Convert normal date -> Excel serial date."""
    date = pd.Timestamp(date)
    return (date - pd.Timestamp("1899-12-30")).days


def d(date):
    return excel_serial(date)


# Example
print("2026-09-01 =", d("2026-09-01"))


## 3. Sample Timesheet DataFrame

In [ ]:
timesheet_df = pd.DataFrame([
    # =========================================================
    # E001 - Technical - Week 1
    # 5 submitted days -> Submitted
    # =========================================================
    {"EmpID": "E001", "DateofEntry": d("2026-09-01"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-02"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-03"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-04"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-05"), "Status": "Submitted"},

    # Multiple entries on the same day.
    # These still count as ONE submitted day.
    {"EmpID": "E001", "DateofEntry": d("2026-09-01"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-01"), "Status": "Submitted"},

    # Rare weekend record - ignored for weekday calculation
    {"EmpID": "E001", "DateofEntry": d("2026-09-06"), "Status": "Submitted"},


    # =========================================================
    # E002 - Technical - Week 1
    # 3 submitted + 2 records with blank status
    # -> Filled but not submitted
    # =========================================================
    {"EmpID": "E002", "DateofEntry": d("2026-09-01"), "Status": "Submitted"},
    {"EmpID": "E002", "DateofEntry": d("2026-09-02"), "Status": "Submitted"},
    {"EmpID": "E002", "DateofEntry": d("2026-09-03"), "Status": "Submitted"},
    {"EmpID": "E002", "DateofEntry": d("2026-09-04"), "Status": None},
    {"EmpID": "E002", "DateofEntry": d("2026-09-05"), "Status": None},


    # =========================================================
    # E003 - Technical - Week 1
    # NO RECORDS
    # -> Blank
    # =========================================================


    # =========================================================
    # E001 - Week 2
    # 5 submitted
    # -> Submitted
    # =========================================================
    {"EmpID": "E001", "DateofEntry": d("2026-09-07"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-08"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-09"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-10"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-11"), "Status": "Submitted"},


    # =========================================================
    # E002 - Week 2
    # 2 submitted + 2 filled but not submitted
    # =========================================================
    {"EmpID": "E002", "DateofEntry": d("2026-09-07"), "Status": "Submitted"},
    {"EmpID": "E002", "DateofEntry": d("2026-09-08"), "Status": "Submitted"},
    {"EmpID": "E002", "DateofEntry": d("2026-09-09"), "Status": None},
    {"EmpID": "E002", "DateofEntry": d("2026-09-10"), "Status": None},


    # =========================================================
    # E003 - Week 2
    # Records exist but nothing submitted
    # -> Filled but not submitted
    # =========================================================
    {"EmpID": "E003", "DateofEntry": d("2026-09-07"), "Status": None},
    {"EmpID": "E003", "DateofEntry": d("2026-09-08"), "Status": None},
    {"EmpID": "E003", "DateofEntry": d("2026-09-09"), "Status": None},
    {"EmpID": "E003", "DateofEntry": d("2026-09-10"), "Status": None},
    {"EmpID": "E003", "DateofEntry": d("2026-09-11"), "Status": None},


    # =========================================================
    # E001 - Week 3
    # EXACTLY 4 submitted
    # -> Filled but not submitted
    # because rule is > 4
    # =========================================================
    {"EmpID": "E001", "DateofEntry": d("2026-09-14"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-15"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-16"), "Status": "Submitted"},
    {"EmpID": "E001", "DateofEntry": d("2026-09-17"), "Status": "Submitted"},


    # =========================================================
    # E003 - Week 3
    # 5 submitted
    # -> Submitted
    # =========================================================
    {"EmpID": "E003", "DateofEntry": d("2026-09-14"), "Status": "Submitted"},
    {"EmpID": "E003", "DateofEntry": d("2026-09-15"), "Status": "Submitted"},
    {"EmpID": "E003", "DateofEntry": d("2026-09-16"), "Status": "Submitted"},
    {"EmpID": "E003", "DateofEntry": d("2026-09-17"), "Status": "Submitted"},
    {"EmpID": "E003", "DateofEntry": d("2026-09-18"), "Status": "Submitted"},


    # =========================================================
    # E004 - Actuarial
    # Used to test team filtering
    # =========================================================
    {"EmpID": "E004", "DateofEntry": d("2026-09-01"), "Status": "Submitted"},
    {"EmpID": "E004", "DateofEntry": d("2026-09-02"), "Status": "Submitted"},
    {"EmpID": "E004", "DateofEntry": d("2026-09-03"), "Status": "Submitted"},
    {"EmpID": "E004", "DateofEntry": d("2026-09-04"), "Status": "Submitted"},
    {"EmpID": "E004", "DateofEntry": d("2026-09-07"), "Status": "Submitted"},
    {"EmpID": "E004", "DateofEntry": d("2026-09-08"), "Status": "Submitted"},
])

timesheet_df


## 4. Full Function: One Employee + Month + Year

In [ ]:
def employee_weekly_timesheet_report(
    timesheet_df,
    emp_id,
    month,
    year
):
    """
    Generate weekly timesheet report for ONE employee.

    Rules:
        submitted_days > 4
            -> Submitted

        records exist AND submitted_days <= 4
            -> Filled but not submitted

        no records
            -> Blank

    Multiple rows on the same employee/date count as ONE day.
    If ANY row on that day has Status='Submitted',
    the day is considered Submitted.
    """

    df = timesheet_df.copy()

    # ---------------------------------------------------------
    # 1. Filter employee
    # ---------------------------------------------------------
    df = df[
        df["EmpID"].astype(str) == str(emp_id)
    ].copy()

    # ---------------------------------------------------------
    # 2. Convert Excel serial date
    # ---------------------------------------------------------
    df["Date"] = pd.to_datetime(
        df["DateofEntry"],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    ).dt.normalize()

    # Remove invalid dates
    df = df[df["Date"].notna()].copy()

    # ---------------------------------------------------------
    # 3. Selected month
    # ---------------------------------------------------------
    month_start = pd.Timestamp(
        year=year,
        month=month,
        day=1
    )

    month_end = (
        month_start
        + pd.offsets.MonthEnd(1)
    )

    # ---------------------------------------------------------
    # 4. Filter selected month
    # ---------------------------------------------------------
    df = df[
        (df["Date"] >= month_start) &
        (df["Date"] <= month_end)
    ].copy()

    # ---------------------------------------------------------
    # 5. Ignore Saturday/Sunday
    # ---------------------------------------------------------
    df = df[
        df["Date"].dt.weekday < 5
    ].copy()

    # ---------------------------------------------------------
    # 6. Determine whether each DAY is submitted
    # ---------------------------------------------------------
    df["IsSubmitted"] = (
        df["Status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("submitted")
    )

    # Multiple rows for same date -> one daily record
    daily = (
        df
        .groupby("Date", as_index=False)
        .agg(
            IsSubmitted=("IsSubmitted", "any")
        )
    )

    daily["DailyStatus"] = np.where(
        daily["IsSubmitted"],
        "Submitted",
        "Filled but not submitted"
    )

    # ---------------------------------------------------------
    # 7. Monday-based week
    # ---------------------------------------------------------
    daily["WeekStart"] = (
        daily["Date"]
        - pd.to_timedelta(
            daily["Date"].dt.weekday,
            unit="D"
        )
    )

    # ---------------------------------------------------------
    # 8. Generate weeks overlapping selected month
    # ---------------------------------------------------------
    business_days = pd.date_range(
        start=month_start,
        end=month_end,
        freq="B"
    )

    week_starts = (
        business_days
        - pd.to_timedelta(
            business_days.weekday,
            unit="D"
        )
    )

    week_starts = (
        pd.Series(week_starts)
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # 9. Build result
    # ---------------------------------------------------------
    result = []

    for i, week_start in enumerate(
        week_starts,
        start=1
    ):

        week_end = (
            week_start
            + pd.Timedelta(days=6)
        )

        # Restrict partial weeks to selected month
        actual_start = max(
            week_start,
            month_start
        )

        actual_end = min(
            week_end,
            month_end
        )

        expected_days = len(
            pd.bdate_range(
                actual_start,
                actual_end
            )
        )

        week_data = daily[
            (daily["Date"] >= actual_start) &
            (daily["Date"] <= actual_end)
        ]

        submitted_days = int(
            week_data["IsSubmitted"].sum()
        )

        filled_not_submitted_days = int(
            (~week_data["IsSubmitted"]).sum()
        )

        total_filled_days = int(
            len(week_data)
        )

        blank_days = max(
            0,
            expected_days - total_filled_days
        )

        # -----------------------------------------------------
        # Weekly classification
        # -----------------------------------------------------
        if submitted_days > 4:
            weekly_status = "Submitted"

        elif total_filled_days > 0:
            weekly_status = "Filled but not submitted"

        else:
            weekly_status = "Blank"

        result.append({
            "EmpID": emp_id,
            "Week": f"Week {i}",
            "WeekStart": actual_start,
            "WeekEnd": actual_end,
            "Expected_Business_Days": expected_days,
            "Submitted_Days": submitted_days,
            "Filled_Not_Submitted_Days":
                filled_not_submitted_days,
            "Total_Filled_Days":
                total_filled_days,
            "Blank_Days": blank_days,
            "Weekly_Status": weekly_status
        })

    return pd.DataFrame(result)


## 5. Run for E001 — September 2026

In [ ]:
report_e001 = employee_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    emp_id="E001",
    month=9,
    year=2026
)

report_e001


### Expected important result

E001 Week 3 has exactly **4 submitted days**.

Because the rule is:

```python
Submitted_Days > 4
```

Week 3 should therefore be:

```text
Filled but not submitted
```


## 6. Run for E002 — September 2026

In [ ]:
report_e002 = employee_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    emp_id="E002",
    month=9,
    year=2026
)

report_e002


## 7. Run for E003 — September 2026

In [ ]:
report_e003 = employee_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    emp_id="E003",
    month=9,
    year=2026
)

report_e003


## 8. Function for ALL Employees from Employee Master

In [ ]:
def all_employees_weekly_timesheet_report(
    timesheet_df,
    employee_df,
    team,
    month,
    year
):
    """
    Generate weekly report for every employee in employee_df
    belonging to the selected team.

    This is important because employees with ZERO timesheet
    records still need to appear as Blank.
    """

    employees = employee_df[
        employee_df["Team"]
        .astype(str)
        .str.strip()
        .str.lower()
        ==
        str(team).strip().lower()
    ].copy()

    employees = employees.drop_duplicates(
        subset=["EmpID"]
    )

    reports = []

    for emp_id in employees["EmpID"]:

        employee_report = employee_weekly_timesheet_report(
            timesheet_df=timesheet_df,
            emp_id=emp_id,
            month=month,
            year=year
        )

        # Add employee information
        employee_info = employees[
            employees["EmpID"] == emp_id
        ].iloc[0]

        employee_report["Name"] = employee_info["Name"]
        employee_report["Team"] = employee_info["Team"]

        reports.append(employee_report)

    if not reports:
        return pd.DataFrame()

    final_report = pd.concat(
        reports,
        ignore_index=True
    )

    # Put employee columns first
    final_report = final_report[
        [
            "EmpID",
            "Name",
            "Team",
            "Week",
            "WeekStart",
            "WeekEnd",
            "Expected_Business_Days",
            "Submitted_Days",
            "Filled_Not_Submitted_Days",
            "Total_Filled_Days",
            "Blank_Days",
            "Weekly_Status"
        ]
    ]

    return final_report


## 9. Run ALL Technical Employees

In [ ]:
technical_report = all_employees_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    employee_df=employee_df,
    team="Technical",
    month=9,
    year=2026
)

technical_report


Notice that **E003 appears even though E003 has no Week 1 timesheet records**.

That is because the employee list comes from `employee_df`, not from the timesheet data. Therefore the employee can correctly be classified as `Blank`.


## 10. Run ALL Actuarial Employees

In [ ]:
actuarial_report = all_employees_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    employee_df=employee_df,
    team="Actuarial",
    month=9,
    year=2026
)

actuarial_report


## 11. Compact Status Report

In [ ]:
technical_status = technical_report[
    [
        "EmpID",
        "Name",
        "Team",
        "Week",
        "Submitted_Days",
        "Filled_Not_Submitted_Days",
        "Blank_Days",
        "Weekly_Status"
    ]
]

technical_status


## 12. Pivot: One Employee per Row, Weeks as Columns

In [ ]:
status_pivot = technical_report.pivot(
    index=["EmpID", "Name", "Team"],
    columns="Week",
    values="Weekly_Status"
).reset_index()

status_pivot


## 13. How to Use With Your Real Data

Once you have your real DataFrames:

```python
employee_df = ...
timesheet_df = ...
```

Run:

```python
report = all_employees_weekly_timesheet_report(
    timesheet_df=timesheet_df,
    employee_df=employee_df,
    team="Technical",
    month=9,
    year=2026
)
```

If your actual employee-name column is different, change `"Name"` inside the function.

The function expects:

### employee_df

```text
EmpID
Name
Team
```

### timesheet_df

```text
EmpID
DateofEntry
Status
```

Your actual `timesheet_df` can contain all the other columns from your screenshot; the function only uses these three columns.


# 14. Resource Manager Dashboard / Graphs

The following visualizations use the **same sample data already created above**.

The goal is not just to show statistics, but to answer:

> **Who needs my attention and who should I email?**

Recommended views:

1. KPI summary
2. Overall status distribution
3. Status by team
4. Employee × Week heatmap
5. Employees requiring action
6. Weekly compliance trend
7. Submitted-day distribution
8. Actionable email lists


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Use the existing technical_report generated above.
dashboard_df = technical_report.copy()

# ------------------------------------------------------------
# KPI summary
# ------------------------------------------------------------
latest_week = dashboard_df["WeekStart"].max()

latest = dashboard_df[
    dashboard_df["WeekStart"] == latest_week
].copy()

total_employees = latest["EmpID"].nunique()
submitted_employees = (latest["Weekly_Status"] == "Submitted").sum()
filled_employees = (
    latest["Weekly_Status"] == "Filled but not submitted"
).sum()
blank_employees = (
    latest["Weekly_Status"] == "Blank"
).sum()

compliance_pct = (
    submitted_employees / total_employees * 100
    if total_employees else 0
)

print("RESOURCE MANAGER KPI SUMMARY")
print("-" * 45)
print("Total Employees              :", total_employees)
print("Submitted                    :", submitted_employees)
print("Filled but not submitted     :", filled_employees)
print("Blank                        :", blank_employees)
print(f"Submission Compliance        : {compliance_pct:.1f}%")


In [ ]:
# ------------------------------------------------------------
# Graph 1: Overall status distribution
# ------------------------------------------------------------

status_counts = latest["Weekly_Status"].value_counts()

plt.figure(figsize=(7, 5))
plt.pie(
    status_counts.values,
    labels=status_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title(f"Timesheet Status Distribution - {latest_week:%d %b %Y}")
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# Graph 2: Status by Team
# ------------------------------------------------------------

team_status = pd.crosstab(
    latest["Team"],
    latest["Weekly_Status"],
    normalize="index"
) * 100

status_order = [
    "Submitted",
    "Filled but not submitted",
    "Blank"
]

team_status = team_status.reindex(
    columns=status_order,
    fill_value=0
)

ax = team_status.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

ax.set_title("Timesheet Status by Team")
ax.set_xlabel("Team")
ax.set_ylabel("Employees (%)")
ax.set_ylim(0, 100)
ax.legend(title="Status", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Employee × Week Heatmap

This is one of the most useful views for a Resource Manager.

- `2` = Submitted
- `1` = Filled but not submitted
- `0` = Blank

The labels below the chart show the actual status names.


In [ ]:
# ------------------------------------------------------------
# Graph 3: Employee x Week heatmap
# ------------------------------------------------------------

status_code = {
    "Blank": 0,
    "Filled but not submitted": 1,
    "Submitted": 2
}

heatmap_df = dashboard_df.copy()
heatmap_df["StatusCode"] = heatmap_df["Weekly_Status"].map(status_code)

heatmap = heatmap_df.pivot_table(
    index=["EmpID", "Name"],
    columns="Week",
    values="StatusCode",
    aggfunc="first"
)

# Keep weeks in natural order
week_order = sorted(
    heatmap.columns,
    key=lambda x: int(str(x).split()[-1])
)
heatmap = heatmap.reindex(columns=week_order)

plt.figure(
    figsize=(
        max(8, len(heatmap.columns) * 1.5),
        max(5, len(heatmap) * 0.55)
    )
)

plt.imshow(
    heatmap.values,
    aspect="auto",
    interpolation="nearest"
)

plt.xticks(
    range(len(heatmap.columns)),
    heatmap.columns
)

plt.yticks(
    range(len(heatmap.index)),
    [f"{emp_id} - {name}" for emp_id, name in heatmap.index]
)

plt.xlabel("Week")
plt.ylabel("Employee")
plt.title("Employee × Week Timesheet Status")

cbar = plt.colorbar()
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels([
    "Blank",
    "Filled but not submitted",
    "Submitted"
])

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# Graph 4: Employees requiring action
# ------------------------------------------------------------

action_counts = (
    dashboard_df
    .assign(
        NeedsAction=dashboard_df["Weekly_Status"] != "Submitted"
    )
    .groupby(["EmpID", "Name"])["NeedsAction"]
    .sum()
    .sort_values(ascending=False)
)

action_counts = action_counts[action_counts > 0]

if len(action_counts) > 0:
    ax = action_counts.sort_values().plot(
        kind="barh",
        figsize=(9, 6)
    )

    ax.set_title("Employees Requiring Action")
    ax.set_xlabel("Non-compliant Weeks")
    ax.set_ylabel("Employee")
    plt.tight_layout()
    plt.show()
else:
    print("No employees require action.")


In [ ]:
# ------------------------------------------------------------
# Graph 5: Weekly compliance trend
# ------------------------------------------------------------

weekly_compliance = (
    dashboard_df
    .assign(
        IsSubmitted=dashboard_df["Weekly_Status"].eq("Submitted")
    )
    .groupby("WeekStart")["IsSubmitted"]
    .mean()
    * 100
)

weekly_compliance.index = [
    f"Week {i + 1}"
    for i in range(len(weekly_compliance))
]

plt.figure(figsize=(9, 5))
plt.plot(
    weekly_compliance.index,
    weekly_compliance.values,
    marker="o"
)

plt.title("Weekly Timesheet Submission Compliance")
plt.xlabel("Week")
plt.ylabel("Submitted Employees (%)")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# Graph 6: Submitted-day distribution
# ------------------------------------------------------------

submitted_day_distribution = (
    latest["Submitted_Days"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(9, 5))
plt.bar(
    submitted_day_distribution.index.astype(str),
    submitted_day_distribution.values
)

plt.title("Distribution of Submitted Days")
plt.xlabel("Submitted Days")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()


# 15. Email Action Lists

These two tables are designed to make the actual email process easier.

### Group 1 — Blank

Employees with no timesheet records for the selected week.

### Group 2 — Filled but not submitted

Employees who have entered timesheet data but have not met the submission threshold.


In [ ]:
# ------------------------------------------------------------
# Email Group 1: Blank employees
# ------------------------------------------------------------

blank_email_list = latest[
    latest["Weekly_Status"] == "Blank"
][
    [
        "EmpID",
        "Name",
        "Team",
        "Week",
        "Weekly_Status"
    ]
].sort_values(["Team", "Name"])

print("BLANK TIMESHEET - EMAIL LIST")
blank_email_list


In [ ]:
# ------------------------------------------------------------
# Email Group 2: Filled but not submitted
# ------------------------------------------------------------

filled_not_submitted_email_list = latest[
    latest["Weekly_Status"] == "Filled but not submitted"
][
    [
        "EmpID",
        "Name",
        "Team",
        "Week",
        "Submitted_Days",
        "Total_Filled_Days",
        "Weekly_Status"
    ]
].sort_values(
    ["Team", "Name"]
)

print("FILLED BUT NOT SUBMITTED - EMAIL LIST")
filled_not_submitted_email_list


## 16. Suggested Manager Workflow

A practical Resource Manager workflow from this dashboard is:

1. Check the **KPI summary**.
2. Check **Status by Team** to identify teams with low compliance.
3. Use the **Employee × Week heatmap** to identify repeated problems.
4. Use **Blank Email List** for employees who have not entered anything.
5. Use **Filled but Not Submitted Email List** for employees who entered data but did not submit.
6. Use **Weekly Compliance Trend** to see whether reminders are improving compliance.

The existing sample data has not been changed; all charts are generated from the same DataFrames and report logic already present in this notebook.


# Final Resource Manager View — All Teams

The default view is **Team = All**.

The main visual is an **Employee × Week Timesheet Compliance Matrix**:

- 🟢 Submitted
- 🟠 Filled but not submitted
- 🔴 Blank

The `Team` column remains visible so the Resource Manager can see every employee and their team in one consolidated report.


In [ ]:

# ============================================================
# FINAL RESOURCE MANAGER FUNCTIONS
# ============================================================

def convert_excel_date(series):
    return pd.to_datetime(
        series,
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )


def prepare_timesheet(timesheet_df):
    df = timesheet_df.copy()

    df["Date"] = convert_excel_date(
        df["DateofEntry"]
    ).dt.normalize()

    df = df[df["Date"].notna()].copy()

    df["Status_Normalized"] = (
        df["Status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["IsSubmitted"] = (
        df["Status_Normalized"] == "submitted"
    )

    return df


def create_weekly_employee_status(
    timesheet_df,
    employee_df,
    month,
    year,
    team="All"
):
    """
    Create weekly status for every employee.

    > 4 submitted days
        -> Submitted

    Records exist but <= 4 submitted days
        -> Filled but not submitted

    No records
        -> Blank
    """

    ts = prepare_timesheet(timesheet_df)
    emp = employee_df.copy()

    if str(team).strip().lower() != "all":
        emp = emp[
            emp["Team"].astype(str).str.strip().str.lower()
            == str(team).strip().lower()
        ].copy()

    emp = emp.drop_duplicates(subset=["EmpID"])

    month_start = pd.Timestamp(
        year=year, month=month, day=1
    )
    month_end = month_start + pd.offsets.MonthEnd(1)

    ts = ts[
        (ts["Date"] >= month_start) &
        (ts["Date"] <= month_end)
    ].copy()

    ts = ts[
        ts["EmpID"].isin(emp["EmpID"])
    ].copy()

    # Ignore weekends.
    ts = ts[ts["Date"].dt.weekday < 5].copy()

    # Multiple rows for one employee/date = one day.
    daily = (
        ts.groupby(["EmpID", "Date"], as_index=False)
          .agg(IsSubmitted=("IsSubmitted", "any"))
    )

    daily["DailyStatus"] = np.where(
        daily["IsSubmitted"],
        "Submitted",
        "Filled but not submitted"
    )

    daily["WeekStart"] = (
        daily["Date"]
        - pd.to_timedelta(daily["Date"].dt.weekday, unit="D")
    )

    # Business weeks overlapping the selected month.
    business_days = pd.date_range(
        start=month_start,
        end=month_end,
        freq="B"
    )

    week_starts = (
        pd.Series(
            business_days
            - pd.to_timedelta(
                business_days.weekday,
                unit="D"
            )
        )
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    # Employee x Week ensures zero-record employees appear as Blank.
    employee_week = (
        emp[["EmpID", "Name", "Team"]]
        .assign(_key=1)
        .merge(
            pd.DataFrame({
                "WeekStart": week_starts,
                "_key": 1
            }),
            on="_key"
        )
        .drop(columns="_key")
    )

    employee_week["WeekEnd"] = (
        employee_week["WeekStart"]
        + pd.Timedelta(days=6)
    )

    employee_week = employee_week[
        (employee_week["WeekEnd"] >= month_start) &
        (employee_week["WeekStart"] <= month_end)
    ].copy()

    employee_week = employee_week.merge(
        daily[["EmpID", "Date", "WeekStart", "DailyStatus"]],
        on=["EmpID", "WeekStart"],
        how="left"
    )

    employee_week = employee_week[
        employee_week["Date"].isna()
        |
        (
            (employee_week["Date"] >= month_start) &
            (employee_week["Date"] <= month_end)
        )
    ].copy()

    employee_week["Submitted_Day"] = (
        employee_week["DailyStatus"] == "Submitted"
    ).astype(int)

    employee_week["Filled_Not_Submitted_Day"] = (
        employee_week["DailyStatus"]
        == "Filled but not submitted"
    ).astype(int)

    summary = (
        employee_week
        .groupby(
            ["EmpID", "Name", "Team", "WeekStart", "WeekEnd"],
            as_index=False
        )
        .agg(
            Submitted_Days=("Submitted_Day", "sum"),
            Filled_Not_Submitted_Days=(
                "Filled_Not_Submitted_Day", "sum"
            )
        )
    )

    def expected_days(row):
        start = max(row["WeekStart"], month_start)
        end = min(row["WeekEnd"], month_end)

        return len(pd.bdate_range(start=start, end=end))

    summary["Expected_Business_Days"] = summary.apply(
        expected_days,
        axis=1
    )

    summary["Total_Filled_Days"] = (
        summary["Submitted_Days"]
        + summary["Filled_Not_Submitted_Days"]
    )

    summary["Blank_Days"] = (
        summary["Expected_Business_Days"]
        - summary["Total_Filled_Days"]
    ).clip(lower=0)

    summary["Weekly_Status"] = np.select(
        [
            summary["Submitted_Days"] > 4,
            summary["Total_Filled_Days"] > 0
        ],
        [
            "Submitted",
            "Filled but not submitted"
        ],
        default="Blank"
    )

    weeks = (
        summary[["WeekStart"]]
        .drop_duplicates()
        .sort_values("WeekStart")
        .reset_index(drop=True)
    )

    weeks["Week"] = [
        f"W{i + 1}" for i in range(len(weeks))
    ]

    summary = summary.merge(
        weeks,
        on="WeekStart",
        how="left"
    )

    return summary.sort_values(
        ["Team", "Name", "WeekStart"]
    ).reset_index(drop=True)


## Run the final report — All Teams

In [ ]:

selected_month = 9
selected_year = 2026
selected_team = "All"

final_report = create_weekly_employee_status(
    timesheet_df=timesheet_df,
    employee_df=employee_df,
    month=selected_month,
    year=selected_year,
    team=selected_team
)

final_report


## Employee × Week Status Matrix

In [ ]:

status_matrix = final_report.pivot_table(
    index=["EmpID", "Name", "Team"],
    columns="Week",
    values="Weekly_Status",
    aggfunc="first"
).reset_index()

week_columns = sorted(
    [c for c in status_matrix.columns if str(c).startswith("W")],
    key=lambda x: int(str(x)[1:])
)

status_matrix = status_matrix[
    ["EmpID", "Name", "Team"] + week_columns
]

display_matrix = status_matrix.copy()

for col in week_columns:
    display_matrix[col] = display_matrix[col].map({
        "Submitted": "🟢",
        "Filled but not submitted": "🟠",
        "Blank": "🔴"
    })

display(display_matrix)


### Legend

🟢 **Submitted** — more than 4 submitted days

🟠 **Filled but not submitted** — records exist, but 4 or fewer submitted days

🔴 **Blank** — no timesheet record


## Resource Manager KPI Summary

In [ ]:

latest_week = final_report["WeekStart"].max()

latest_report = final_report[
    final_report["WeekStart"] == latest_week
].copy()

total_employees = len(latest_report)

submitted_count = (
    latest_report["Weekly_Status"] == "Submitted"
).sum()

filled_count = (
    latest_report["Weekly_Status"]
    == "Filled but not submitted"
).sum()

blank_count = (
    latest_report["Weekly_Status"] == "Blank"
).sum()

compliance = (
    submitted_count / total_employees * 100
    if total_employees else 0
)

print("RESOURCE MANAGER SUMMARY")
print("=" * 50)
print(f"Team selected        : {selected_team}")
print(f"Month                : {selected_month}/{selected_year}")
print(f"Total employees      : {total_employees}")
print(f"Submitted            : {submitted_count}")
print(f"Filled not submitted : {filled_count}")
print(f"Blank                : {blank_count}")
print(f"Compliance            : {compliance:.1f}%")


## Employees Requiring Action

In [ ]:

action_list = latest_report[
    latest_report["Weekly_Status"] != "Submitted"
][
    [
        "EmpID",
        "Name",
        "Team",
        "Week",
        "Submitted_Days",
        "Total_Filled_Days",
        "Blank_Days",
        "Weekly_Status"
    ]
].sort_values(
    ["Weekly_Status", "Team", "Name"]
)

display(action_list)


## Separate Email Lists

In [ ]:

blank_email_list = latest_report[
    latest_report["Weekly_Status"] == "Blank"
][
    ["EmpID", "Name", "Team", "Week"]
].sort_values(["Team", "Name"])

filled_not_submitted_email_list = latest_report[
    latest_report["Weekly_Status"] == "Filled but not submitted"
][
    [
        "EmpID",
        "Name",
        "Team",
        "Week",
        "Submitted_Days",
        "Total_Filled_Days"
    ]
].sort_values(["Team", "Name"])

print("=== BLANK TIMESHEET EMAIL LIST ===")
display(blank_email_list)

print("=== FILLED BUT NOT SUBMITTED EMAIL LIST ===")
display(filled_not_submitted_email_list)


## Optional Team Filter

The dashboard defaults to:

```python
selected_team = "All"
```

To view only one team:

```python
selected_team = "Technical"
```

or:

```python
selected_team = "Actuarial"
```

The same report logic is used; only the employee population changes.


## Export Employee × Week Matrix to PNG
Status legend: 🟢 Submitted | 🟡 Filled but not submitted | 🔴 Blank

In [ ]:
# Export Employee × Week Matrix as a PNG with colored status boxes
# 🟢 Submitted | 🟡 Filled but not submitted | 🔴 Blank

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

# Keep the original status values for determining box colors
matrix = status_matrix.copy()

# Remove any existing emoji if this cell is run after a previous conversion
matrix = matrix.replace({
    "🟢": "Submitted",
    "🟡": "Filled but not submitted",
    "🔴": "Blank"
})

status_colors = {
    "Submitted": "#2E7D32",                 # Green
    "Filled but not submitted": "#F9A825",  # Yellow
    "Blank": "#D32F2F"                       # Red
}

fig_width = max(12, len(matrix.columns) * 1.7)
fig_height = max(4, len(matrix) * 0.55 + 1)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))
ax.axis("off")

# Table layout
n_rows, n_cols = matrix.shape
table = ax.table(
    cellText=matrix.astype(str).values,
    colLabels=matrix.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Style all cells
for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(0.8)

    # Header row
    if row == 0:
        cell.set_facecolor("#E0E0E0")
        cell.get_text().set_weight("bold")
        cell.get_text().set_color("black")

    # Data rows
    else:
        column_name = matrix.columns[col]
        value = matrix.iloc[row - 1, col]

        # Only weekly status columns become colored boxes.
        if column_name not in ["EmpID", "Name", "Team"]:
            if value in status_colors:
                cell.set_facecolor(status_colors[value])
                cell.get_text().set_text("")
                cell.set_edgecolor("white")
                cell.set_linewidth(2.0)
        else:
            cell.set_facecolor("white")

plt.tight_layout()
plt.savefig(
    "employee_weekly_matrix.png",
    dpi=200,
    bbox_inches="tight",
    facecolor="white"
)
plt.close()

print("PNG created: employee_weekly_matrix.png")
